## Sel 1: Import Pustaka

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

import mlflow
import dagshub

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb

from sklearn.multioutput import MultiOutputRegressor

import joblib
import contextlib
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

## Sel 2: Memuat & Pra-pemrosesan Data (Resampling Harian)

In [2]:
# Pastikan path ini sesuai dengan posisi .env milikmu
load_dotenv('../.env', override=True)

# Ambil URL MLflow utuh 
db_url_mlflow = os.getenv("DATABASE_URL_DIRECT")

if not db_url_mlflow:
    raise ValueError("DATABASE_URL_DIRECT tidak ditemukan di .env!")

# Buat URL polos khusus untuk Pandas & SQLAlchemy
db_url_pandas = db_url_mlflow.split("?")[0]

print("Membuka jembatan ke Supabase...")
# Gunakan URL polos untuk engine
engine = create_engine(db_url_pandas)

# ==========================================
# LANGKAH 1: KUERI SQL JATAH RATA
# ==========================================
RENTANG_WAKTU = "1 year"

query_sql = f"""
    SELECT 
        waktu_aktual, 
        id_wilayah, 
        pm25, pm10, so2, co, no2, ozon
    FROM public.data_historis
    WHERE waktu_aktual >= NOW() - INTERVAL '{RENTANG_WAKTU}';
"""

ukuran_chunk = 50000
chunks = []

with engine.connect() as conn:
    conn.execute(text("SET statement_timeout = 0"))
    conn.commit() # Wajib commit agar pengaturannya tersimpan di sesi ini
    
    stream_conn = conn.execution_options(stream_results=True)
    
    for i, chunk in enumerate(pd.read_sql(text(query_sql), stream_conn, chunksize=ukuran_chunk)):
        print(f"✅ Berhasil menyedot batch ke-{i+1} (hingga {(i+1) * ukuran_chunk} baris...)")
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

kolom_waktu = 'waktu_aktual'
kolom_kota = 'id_wilayah'  
polutan = ['pm25', 'pm10', 'so2', 'co', 'no2', 'ozon'] 

# PRA-PEMROSESAN & RESAMPLE
df[kolom_waktu] = pd.to_datetime(df[kolom_waktu])
df[kolom_waktu] = df[kolom_waktu].dt.tz_localize(None)
df.set_index(kolom_waktu, inplace=True)

df_daily = df.groupby(kolom_kota)[polutan].resample('D').mean().reset_index()

# ==========================================
# LANGKAH 2: INTERPOLASI (Pengganti dropna)
# ==========================================
print("Menambal data sensor yang bolong dengan interpolasi...")

df_daily = df_daily.groupby(kolom_kota, group_keys=False).apply(
    lambda group: group.interpolate(method='linear')
).reset_index(drop=True)

df_daily = df_daily.bfill().ffill()

print(f"✅ Selesai! Bentuk data akhir yang adil & solid: {df_daily.shape}")
display(df_daily.head())

Membuka jembatan ke Supabase...
✅ Berhasil menyedot batch ke-1 (hingga 50000 baris...)
✅ Berhasil menyedot batch ke-2 (hingga 100000 baris...)
✅ Berhasil menyedot batch ke-3 (hingga 150000 baris...)
✅ Berhasil menyedot batch ke-4 (hingga 200000 baris...)
✅ Berhasil menyedot batch ke-5 (hingga 250000 baris...)
✅ Berhasil menyedot batch ke-6 (hingga 300000 baris...)
✅ Berhasil menyedot batch ke-7 (hingga 350000 baris...)
Menambal data sensor yang bolong dengan interpolasi...
✅ Selesai! Bentuk data akhir yang adil & solid: (13870, 8)


,id_wilayah,waktu_aktual,pm25,pm10,so2,co,no2,ozon
0,1,2025-05-19,3.851429,7.954286,0.280000,104.245714,1.018571,43.594286
1,1,2025-05-20,4.512917,7.609167,0.371667,120.107500,1.085417,46.918750
2,1,2025-05-21,4.322500,7.115417,0.307917,110.498750,1.011667,46.697500
3,1,2025-05-22,6.497917,10.666250,0.480417,141.382083,1.296250,47.882500
4,1,2025-05-23,9.252083,11.143750,1.937083,360.070833,9.375417,28.992083


## Sel 3: Rekayasa Fitur (Sliding Window 3 Hari & Target Besok)

In [3]:
# Fungsi untuk membuat Sliding Window, Fitur Temporal, dan Rolling Stats
def create_advanced_features(df_kota, n_lags=3):
    df_temp = df_kota.copy()
    
    # 🌟 FITUR BARU 1: KONTEKS WAKTU (TEMPORAL)
    df_temp['Bulan'] = df_temp[kolom_waktu].dt.month
    df_temp['Is_Weekend'] = df_temp[kolom_waktu].dt.dayofweek.isin([5, 6]).astype(int)

    for p in polutan:
        # --- FITUR LAMA: HISTORY (Kemarin t-1, Lusa t-2, dst) ---
        for i in range(1, n_lags + 1):
            df_temp[f'{p}_H-{i}'] = df_temp[p].shift(i)
            
        # 🌟 FITUR BARU 2: TREN JANGKA MENENGAH (ROLLING STATS)
        # PERBAIKAN: Menghapus .shift(1) sebelum rolling agar nilai HARI INI (t) 
        # ikut dihitung dalam rata-rata 3 hari terakhir (t, t-1, t-2).
        # Ini aman dari data leakage karena target kita adalah BESOK (t+1).
        df_temp[f'{p}_RollMean_3'] = df_temp[p].rolling(window=3).mean()
        df_temp[f'{p}_RollMax_3'] = df_temp[p].rolling(window=3).max()
        
        # --- TARGET BESOK (t+1) ---
        df_temp[f'TARGET_{p}_Besok'] = df_temp[p].shift(-1)
        
    return df_temp

print("Meracik fitur tingkat lanjut...")

# Aplikasikan fungsi di atas untuk masing-masing kota/wilayah secara terpisah
df_model = df_daily.groupby(kolom_kota, group_keys=False).apply(create_advanced_features)

# Hapus baris yang memiliki NaN akibat pergeseran history, rolling, dan target besok
df_model = df_model.dropna().reset_index(drop=True)

# Lakukan One-Hot Encoding untuk kolom id_wilayah (kolom_kota)
df_model[kolom_kota] = df_model[kolom_kota].astype(str)
df_model = pd.get_dummies(df_model, columns=[kolom_kota])

# 1. Kumpulkan semua nama kolom target (6 Polutan Besok)
kolom_target = [col for col in df_model.columns if 'TARGET_' in col]

# 2. Pisahkan tabel jawaban (y)
y = df_model[kolom_target]

# 3. Pisahkan tabel soal (X) 
# PERBAIKAN: Membuang kolom teks 'kategori_ispu' dari tabel X (jika ada)
# agar algoritma AI tidak crash karena mencoba membaca tipe data string/objek.
kolom_bawaan_yg_harus_dibuang = [kolom_waktu, 'kategori_ispu']
X = df_model.drop(columns=kolom_target + kolom_bawaan_yg_harus_dibuang, errors='ignore')

print(f"Bentuk tabel awal df_model : {df_model.shape}")
print(f"Dimensi X (Fitur AI BARU!) : {X.shape}")
print(f"Dimensi y (Target AI)      : {y.shape}")

print("\nSanity Check: 5 Baris Pertama dari Tabel Soal (X):")
display(X.head())

print("\nSanity Check: 5 Baris Pertama dari Tabel Jawaban (y):")
display(y.head())

Meracik fitur tingkat lanjut...
Bentuk tabel awal df_model : (13718, 83)
Dimensi X (Fitur AI BARU!) : (13718, 76)
Dimensi y (Target AI)      : (13718, 6)

Sanity Check: 5 Baris Pertama dari Tabel Soal (X):


,pm25,pm10,so2,co,no2,ozon,Bulan,Is_Weekend,pm25_H-1,pm25_H-2,...,id_wilayah_35,id_wilayah_36,id_wilayah_37,id_wilayah_38,id_wilayah_4,id_wilayah_5,id_wilayah_6,id_wilayah_7,id_wilayah_8,id_wilayah_9
0,6.497917,10.666250,0.480417,141.382083,1.296250,47.882500,5,0,4.322500,4.512917,...,False,False,False,False,False,False,False,False,False,False
1,9.252083,11.143750,1.937083,360.070833,9.375417,28.992083,5,0,6.497917,4.322500,...,False,False,False,False,False,False,False,False,False,False
2,43.887917,48.834167,3.260417,830.073750,19.633750,60.451250,5,1,9.252083,6.497917,...,False,False,False,False,False,False,False,False,False,False
3,57.849167,65.500833,4.447500,928.293750,11.708750,70.673333,5,1,43.887917,9.252083,...,False,False,False,False,False,False,False,False,False,False
4,37.407500,43.739583,3.179167,831.830833,16.557917,26.980417,5,0,57.849167,43.887917,...,False,False,False,False,False,False,False,False,False,False



Sanity Check: 5 Baris Pertama dari Tabel Jawaban (y):


,TARGET_pm25_Besok,TARGET_pm10_Besok,TARGET_so2_Besok,TARGET_co_Besok,TARGET_no2_Besok,TARGET_ozon_Besok
0,9.252083,11.143750,1.937083,360.070833,9.375417,28.992083
1,43.887917,48.834167,3.260417,830.073750,19.633750,60.451250
2,57.849167,65.500833,4.447500,928.293750,11.708750,70.673333
3,37.407500,43.739583,3.179167,831.830833,16.557917,26.980417
4,25.114583,29.633333,2.150833,505.297083,6.969583,55.860417


## Sel 4: Splitting & Training Baseline Model MLFLOW

In [4]:
# 1. Splitting Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# --- MLFlow DagsHub ---
# Tarik data dari .env
load_dotenv('../.env', override=True)
repo_owner = os.getenv("DAGSHUB_REPO_OWNER")
repo_name = os.getenv("DAGSHUB_REPO_NAME")

if not repo_owner or not repo_name:
    raise ValueError("Identitas DagsHub belum di-set di file .env!")

print(f"Menyiapkan pelacakan MLflow ke server DagsHub ({repo_owner}/{repo_name})...")

# Inisialisasi menggunakan variabel dinamis
dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)

# Nama eksperimenmu
mlflow.set_experiment("01_Baseline_Selection")
# 3. Persiapan Baseline Model 
baselines = {
    "Baseline_LinearRegression": MultiOutputRegressor(LinearRegression(n_jobs=-1)),
    
    # Turunkan n_estimators dari 50 menjadi 10
    "Baseline_RandomForest": MultiOutputRegressor(RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1)),
    
    "Baseline_KNeighbors": MultiOutputRegressor(KNeighborsRegressor(n_jobs=-1)),
    
    # Tambahkan n_estimators=10 secara eksplisit (defaultnya 100)
    "Baseline_XGBoost": MultiOutputRegressor(xgb.XGBRegressor(n_estimators=30, random_state=42, n_jobs=-1))
}

print("--- MEMULAI TRACKING BASELINE MODELS DI MLFLOW ---\n")
for nama_model, model in baselines.items():
    with mlflow.start_run(run_name=nama_model):
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        mlflow.log_param("algoritma", nama_model)
        mlflow.log_metric("test_mae", mae)
        mlflow.log_metric("test_rmse", rmse)
        mlflow.log_metric("test_r2", r2)
        
        print(f"✅ {nama_model:<25} berhasil dicatat (RMSE: {rmse:.2f})")

Menyiapkan pelacakan MLflow ke server DagsHub (tegarkusuma12/Web-ISPU)...


Accessing as tegarkusuma12

Initialized MLflow to track repo "tegarkusuma12/Web-ISPU"

Repository tegarkusuma12/Web-ISPU initialized!

--- MEMULAI TRACKING BASELINE MODELS DI MLFLOW ---

✅ Baseline_LinearRegression berhasil dicatat (RMSE: 19022127.00)
🏃 View run Baseline_LinearRegression at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/0/runs/254b8fff94e5444283918f14e6867802
🧪 View experiment at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/0
✅ Baseline_RandomForest     berhasil dicatat (RMSE: 19.45)
🏃 View run Baseline_RandomForest at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/0/runs/80048de5fb2e4d9fb11a1dae2bda4808
🧪 View experiment at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/0


  File "c:\Users\Tigro\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Tigro\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Tigro\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\Tigro\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


✅ Baseline_KNeighbors       berhasil dicatat (RMSE: 23.58)
🏃 View run Baseline_KNeighbors at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/0/runs/ce28566b457349d3bd55d310b406b59d
🧪 View experiment at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/0
✅ Baseline_XGBoost          berhasil dicatat (RMSE: 19.99)
🏃 View run Baseline_XGBoost at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/0/runs/0b6ffaadd8d84a24ae0368cec5b0b1d2
🧪 View experiment at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/0


## Sel 5 : Hyperparameter Tuning XGBoost MLFlow 

In [ ]:
# print("Memulai Hyperparameter Tuning XGBoost (Strategi Pisahkan Otak)...")

# 1. Siapkan 'brankas' untuk menyimpan 6 otak AI yang berbeda
model_terbaik_per_polutan = {}

# 2. Ruang Pencarian Grid
param_grid_murni = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

# --- JEMBATAN TQDM & JOBLIB ---
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
# ------------------------------------------

total_mae, total_rmse = 0, 0
daftar_kolom_target = y.columns 

# 3. Lakukan Perulangan: Latih 1 AI untuk tiap 1 Polutan
for nama_kolom in daftar_kolom_target:
    
    # MEMBERSIHKAN NAMA UNTUK MLFLOW & PRINT (Kini jauh lebih simpel!)
    # Mengubah 'TARGET_pm25_Besok' menjadi 'pm25'
    nama_polutan_bersih = nama_kolom.replace('TARGET_', '').replace('_Besok', '')
    
    print(f"\n======================================================")
    print(f"🚀 Mendidik AI Spesialis Khusus: {nama_polutan_bersih.upper()}")
    print(f"======================================================")
    
    # Ambil kolom jawaban (y) HANYA untuk gas ini
    y_train_tunggal = y_train[nama_kolom]
    y_test_tunggal = y_test[nama_kolom]
    
    # Inisiasi XGBoost Murni
    model_xgb_dasar = xgb.XGBRegressor(random_state=42)
    
    # Siapkan Mesin Pencari
    grid_search = RandomizedSearchCV(
        estimator=model_xgb_dasar, 
        param_distributions=param_grid_murni, 
        n_iter=15, 
        cv=4,      
        scoring='neg_mean_absolute_error', 
        random_state=42,
        n_jobs=-1, 
        verbose=0  
    )
    
    # [MLFLOW] Membuka Sesi Pencatatan untuk Polutan Spesifik
    with mlflow.start_run(run_name=f"Spesialis_XGBoost_{nama_polutan_bersih.upper()}"):
        
        total_fits = grid_search.n_iter * grid_search.cv 
        with tqdm_joblib(tqdm(desc=f"Tuning {nama_polutan_bersih.upper()}", total=total_fits, unit="fit")):
            grid_search.fit(X_train, y_train_tunggal)
            
        # Ambil otak terbaiknya
        otak_spesialis = grid_search.best_estimator_
        
        # Simpan ke dalam brankas dictionary
        model_terbaik_per_polutan[nama_kolom] = otak_spesialis
        
        print(f"Kombinasi Terbaik:\n{grid_search.best_params_}")
        
        # Lakukan ujian evaluasi langsung
        y_pred_tunggal = otak_spesialis.predict(X_test)
        
        mae = mean_absolute_error(y_test_tunggal, y_pred_tunggal)
        rmse = np.sqrt(mean_squared_error(y_test_tunggal, y_pred_tunggal))
        rata_rata_asli = y_test_tunggal.mean()
        persentase_error = (mae / rata_rata_asli) * 100 if rata_rata_asli > 0 else 0
        
        total_mae += mae
        total_rmse += rmse
        
        print(f"  - Rata-rata asli : {rata_rata_asli:.2f}")
        print(f"  - MAE (Meleset)  : {mae:.2f}")
        print(f"  - RMSE           : {rmse:.2f}")
        print(f"  - Error (%)      : {persentase_error:.2f}%")
        
        # [MLFLOW] Catat skor evaluasi ke dashboard
        mlflow.log_metric("test_mae", mae)
        mlflow.log_metric("test_rmse", rmse)
        
        # [MLFLOW] Catat resep parameter yang menang ke dashboard
        for param_name, param_val in grid_search.best_params_.items():
            mlflow.log_param(param_name, param_val)

print("\n--- 🏁 SEMUA 6 AI SPESIALIS SELESAI DILATIH ---")
print(f"Rata-rata MAE Keseluruhan : {total_mae/6:.2f}")

Memulai Hyperparameter Tuning XGBoost (Strategi Pisahkan Otak)...

🚀 Mendidik AI Spesialis Khusus: PM25


Tuning PM25:   0%|          | 0/60 [00:00<?, ?fit/s]

Kombinasi Terbaik:
{'subsample': 1.0, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
  - Rata-rata asli : 5.28
  - MAE (Meleset)  : 1.56
  - RMSE           : 3.10
  - Error (%)      : 29.64%

🚀 Mendidik AI Spesialis Khusus: PM10


Tuning PM10:   0%|          | 0/60 [00:00<?, ?fit/s]

Kombinasi Terbaik:
{'subsample': 1.0, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
  - Rata-rata asli : 10.85
  - MAE (Meleset)  : 2.58
  - RMSE           : 4.05
  - Error (%)      : 23.80%

🚀 Mendidik AI Spesialis Khusus: SO2


...

Kombinasi Terbaik:
{'subsample': 0.9, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
  - Rata-rata asli : 119.16
  - MAE (Meleset)  : 21.74
  - RMSE           : 41.98
  - Error (%)      : 18.24%

🚀 Mendidik AI Spesialis Khusus: NO2


Tuning NO2:   0%|          | 0/60 [00:00<?, ?fit/s]

Kombinasi Terbaik:
{'subsample': 0.9, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
  - Rata-rata asli : 0.34
  - MAE (Meleset)  : 0.21
  - RMSE           : 0.47
  - Error (%)      : 61.49%

🚀 Mendidik AI Spesialis Khusus: OZON


Tuning OZON:   0%|          | 0/60 [00:00<?, ?fit/s]

Kombinasi Terbaik:
{'subsample': 1.0, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
  - Rata-rata asli : 39.57
  - MAE (Meleset)  : 3.63
  - RMSE           : 5.71
  - Error (%)      : 9.18%

--- 🏁 SEMUA 6 AI SPESIALIS SELESAI DILATIH ---
Rata-rata MAE Keseluruhan : 4.96


## Sel 6: Audit Keadilan Per Wilayah

In [17]:
# =====================================================================
# 1. TABEL STATISTIK AKURASI KESELURUHAN POLUTAN (SETELAH TUNING)
# =====================================================================
print("--- 📊 TABEL STATISTIK AKURASI TIAP POLUTAN (SETELAH TUNING) ---")

statistik_polutan = []

# Looping ke semua model spesialis polutan yang sudah di-training
for polutan, model in model_terbaik_per_polutan.items():
    # Lakukan prediksi
    y_pred = model.predict(X_test)
    y_true = y_test[polutan]
    
    # Kalkulasi Metrik Evaluasi
    rata_rata_asli = y_true.mean()
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Cegah error pembagian dengan nol
    persentase_error = (mae / rata_rata_asli) * 100 if rata_rata_asli != 0 else 0
    
    # Masukkan hasil ke dalam list
    statistik_polutan.append({
        "Polutan": polutan.upper(),
        "Rata-rata Asli": f"{rata_rata_asli:.2f}",
        "MAE (Meleset)": f"{mae:.2f}",
        "RMSE": f"{rmse:.2f}",
        "Error (%)": f"{persentase_error:.2f}%"
    })

# Ubah list menjadi DataFrame Pandas agar tampilannya berbentuk tabel cantik di Jupyter
df_statistik = pd.DataFrame(statistik_polutan)
display(df_statistik) 

print("\n" + "="*60 + "\n")

# =====================================================================
# 2. AUDIT KEADILAN AI PER WILAYAH (Khusus PM2.5)
# =====================================================================
print("--- ⚖️ AUDIT KEADILAN AI PER WILAYAH (Khusus PM2.5) ---")

# Ambil spesialis PM2.5 dari brankas (sesuaikan nama kunci dictionary-nya)
kunci_pm25 = [k for k in model_terbaik_per_polutan.keys() if 'pm25' in k][0]
model_pm25 = model_terbaik_per_polutan[kunci_pm25]

# Lakukan prediksi untuk ujian tes
y_pred_pm25 = model_pm25.predict(X_test)
y_test_pm25 = y_test[kunci_pm25]

# Cari semua kolom hasil One-Hot Encoding untuk wilayah (misal: id_wilayah_1, id_wilayah_2, dst)
kolom_wilayah = [col for col in X_test.columns if col.startswith(f'{kolom_kota}_')]

# Lakukan loop untuk mengecek akurasi di tiap kota
for wil in kolom_wilayah:
    # Filter hanya data yang barisnya milik kota ini (nilai One-Hot = 1 atau True)
    mask = X_test[wil] == 1 
    
    if mask.sum() > 0: # Pastikan kotanya ada di data test
        y_test_wilayah = y_test_pm25[mask]
        y_pred_wilayah = y_pred_pm25[mask]
        
        # Hitung Error (Meleset berapa angka)
        mae_wilayah = mean_absolute_error(y_test_wilayah, y_pred_wilayah)
        
        # Bersihkan nama untuk diprint (ambil ID angkanya saja)
        id_asli = wil.replace(f'{kolom_kota}_', '')
        
        print(f"ID Wilayah {id_asli:<3} | MAE PM2.5: {mae_wilayah:.2f}")

--- 📊 TABEL STATISTIK AKURASI TIAP POLUTAN (SETELAH TUNING) ---


,Polutan,Rata-rata Asli,MAE (Meleset),RMSE,Error (%)
0,TARGET_PM25_BESOK,5.28,1.56,3.10,29.64%
1,TARGET_PM10_BESOK,10.85,2.58,4.05,23.80%
2,TARGET_SO2_BESOK,0.17,0.06,0.13,34.48%
3,TARGET_CO_BESOK,119.16,21.74,41.98,18.24%
4,TARGET_NO2_BESOK,0.34,0.21,0.47,61.49%
5,TARGET_OZON_BESOK,39.57,3.63,5.71,9.18%




--- ⚖️ AUDIT KEADILAN AI PER WILAYAH (Khusus PM2.5) ---
ID Wilayah 31  | MAE PM2.5: 1.06
ID Wilayah 32  | MAE PM2.5: 1.46
ID Wilayah 33  | MAE PM2.5: 1.64
ID Wilayah 34  | MAE PM2.5: 1.13
ID Wilayah 35  | MAE PM2.5: 1.42
ID Wilayah 36  | MAE PM2.5: 1.77
ID Wilayah 37  | MAE PM2.5: 2.16
ID Wilayah 38  | MAE PM2.5: 1.67


## Sel 7: Ekspor Model (Menyimpan Hasil)

In [18]:
# Membungkus KE-ENAM model spesialis dan daftar nama fitur
paket_model = {
    'dict_model_spesialis': model_terbaik_per_polutan,
    'fitur': list(X_train.columns)
}

# Simpan langsung ke kandang DVC (tanpa MLflow log_artifact)
nama_file = 'xgb_ispu_jatim_multi_otak.pkl'
joblib.dump(paket_model, nama_file)

print(f"📦 Selesai! Model pintar telah diekspor sebagai '{nama_file}'")

📦 Selesai! Model pintar telah diekspor sebagai 'xgb_ispu_jatim_multi_otak.pkl'
